# Preprocesamiento de texto - Conversa AI
## Limpieza lingüística y preparación para modelos NLP

**Propósito:** Tomar el DataFrame validado y limpiar los textos en español y portugués, unificarlos en una columna `texto_clean`, y asignar idioma detectado.

**Input:** `data_limpia.parquet` (o el DataFrame resultante de ingest.ipynb)  
**Output:** DataFrame con columnas adicionales: `idioma`, `texto_clean`

**Autor:** Data Engineer  
**Fecha:** Mayo 2026

### 1.Setup y dependencias

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
import os

try:
    from unidecode import unidecode
    USE_UNIDECODE = True
except ImportError:
    USE_UNIDECODE = False
    print("⚠️ unidecode no instalado. Se usará método propio para eliminar acentos.")

print("Librerías cargadas.")

Librerías cargadas.


### 2. Cargar datos limpios (desde ingesta)

In [2]:
# encontrar ruta de data_limpia.parquet

potential_paths = [
	"../data/interim/data_limpia.parquet",
	"../data/interim/data_limpia.parquet",
	"../../data/interim/data_limpia.parquet",
	"data/interim/data_limpia.parquet"
]
for path in potential_paths:
	if os.path.exists(path):
		FILE_PATH = path
		print(f"Archivo encontrado: {FILE_PATH}")
		break

Archivo encontrado: ../data/interim/data_limpia.parquet


In [3]:
# Ajusta la ruta según donde guardaste el output de ingest.ipynb
INPUT_PATH = "../data/interim/data_limpia.parquet"  # o "../data/interim/data_limpia.parquet"

try:
    df = pd.read_parquet(INPUT_PATH)
    print(f"Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")
    display(df.head(2))
except FileNotFoundError:
    # Si no existe parquet, intenta leer el CSV original (pero ya deberías tener el limpio)
    print("Archivo parquet no encontrado. Cargando CSV original...")
    df = pd.read_csv("..data_engineering/data/final/corpus_v2_final.csv")
    print("⚠️ Usando CSV original sin validaciones previas. Se recomienda ejecutar ingest.ipynb primero.")

Datos cargados: 20001 filas, 11 columnas


,session_id,turn_number,flow_name,usuario,fecha,intencion,nivel_frustracion,texto_espanol,texto_portugues,es_churn_risk,resolved
0,SESS-447996,1,Facturación y Cobros,@soyMarely89,2026-01-04 16:50:00,problema_pago,0,Tengo un problema con un cobro.,Tenho um problema com uma cobrança.,0,0
1,SESS-447996,2,Facturación y Cobros,@soyMarely89,2026-01-04 16:55:00,problema_pago,1,Nadie me ayuda con el cobro doble.,Ninguém me ajuda com a cobrança dupla.,0,0


### 3. Asegurar formato de fecha (consistencia)

In [4]:
# Aunque ingest.py ya convirtió, reforzamos (idempotente)
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
    print(f"Rango de fechas: {df['fecha'].min()} a {df['fecha'].max()}")
else:
    print("⚠️ Columna 'fecha' no encontrada.")

Rango de fechas: 2025-11-08 00:58:48 a 2026-05-07 00:52:07


### 4. Asignar idioma basado en columna de texto no nula

In [5]:
# Crear columna 'idioma' según qué texto está presente
def assign_language(row):
    if pd.notna(row.get('texto_espanol')):
        return 'es'
    elif pd.notna(row.get('texto_portugues')):
        return 'pt'
    else:
        return 'unknown'

df['idioma'] = df.apply(assign_language, axis=1)

# Ver distribución
print(df['idioma'].value_counts())
display(df[df['idioma'] == 'unknown'][['texto_espanol', 'texto_portugues']].head())

idioma
es    20001
Name: count, dtype: int64


,texto_espanol,texto_portugues


### 5. Función para normalizar texto (minúsculas, puntuación, acentos opcionales)


In [6]:
def normalize_text(text, remove_accents=True, lower=True, remove_punct=True):
    """
    Limpia texto para NLP.
    
    Parámetros:
    - text: string o NaN
    - remove_accents: eliminar acentos y diacríticos
    - lower: convertir a minúsculas
    - remove_punct: eliminar puntuación (.,!?;: etc.) y caracteres especiales
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    if lower:
        text = text.lower()
    
    if remove_accents:
        if USE_UNIDECODE:
            text = unidecode(text)
        else:
            # Normalización Unicode para separar acentos
            text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    
    if remove_punct:
        # Eliminar cualquier cosa que no sea letra, número o espacio
        text = re.sub(r'[^\w\s]', '', text)
        # Eliminar espacios múltiples
        text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Prueba rápida
ejemplo = "¡Hola! ¿Cómo estás? Isso é um teste: áçãõ."
print("Original:", ejemplo)
print("Normalizado:", normalize_text(ejemplo))

Original: ¡Hola! ¿Cómo estás? Isso é um teste: áçãõ.
Normalizado: hola como estas isso e um teste acao


### 6. Aplicar limpieza a cada texto según idioma

In [7]:
# Creamos una columna 'texto_original' que toma el texto del idioma correspondiente
def select_text_by_language(row):
    if row['idioma'] == 'es':
        return row.get('texto_espanol', '')
    elif row['idioma'] == 'pt':
        return row.get('texto_portugues', '')
    else:
        return ''

df['texto_original'] = df.apply(select_text_by_language, axis=1)

# Aplicar normalización (puedes ajustar parámetros)
df['texto_clean'] = df['texto_original'].apply(
    lambda x: normalize_text(x, remove_accents=True, lower=True, remove_punct=True)
)

# Ver resultados
print("Ejemplos de limpieza:")
display(df[['texto_original', 'texto_clean']].head(5))

Ejemplos de limpieza:


,texto_original,texto_clean
0,Tengo un problema con un cobro.,tengo un problema con un cobro
1,Nadie me ayuda con el cobro doble.,nadie me ayuda con el cobro doble
2,Tengo un problema con un cobro.,tengo un problema con un cobro
3,Quiero cambiar mi plan.,quiero cambiar mi plan
4,El sistema no me deja actualizar.,el sistema no me deja actualizar


### 7. Validación opcional de texto vacío después de limpieza

In [8]:
# Identificar mensajes que quedaron vacíos (solo emojis, puntuación, etc.)
df['texto_vacio'] = df['texto_clean'].str.strip() == ''

print(f"Mensajes vacíos después de limpieza: {df['texto_vacio'].sum()}")
if df['texto_vacio'].sum() > 0:
    print("Ejemplos de originales que resultaron vacíos:")
    display(df[df['texto_vacio']][['texto_original', 'texto_clean']].head(3))

Mensajes vacíos después de limpieza: 0


### 8. Guardar dataset preprocesado

In [9]:
OUTPUT_PATH = "../data/processed/data_preprocessed.parquet"
df.to_parquet(OUTPUT_PATH, index=False)
print(f"Datos preprocesados guardados en {OUTPUT_PATH}")

# También guardamos una copia en CSV para fácil inspección (opcional)
df[['session_id', 'turn_number', 'idioma', 'texto_clean', 'nivel_frustracion', 'intencion', 'resolved']].to_csv("data_preprocesado_sample.csv", index=False)
print("Muestra guardada en CSV.")

Datos preprocesados guardados en ../data/processed/data_preprocessed.parquet
Muestra guardada en CSV.


### 9. Resumen final

In [10]:
print("=== RESUMEN PREPROCESAMIENTO ===")
print(f"Total registros: {len(df)}")
print(f"Idiomas: español={sum(df['idioma']=='es')}, portugués={sum(df['idioma']=='pt')}, desconocido={sum(df['idioma']=='unknown')}")
print(f"Mensajes con texto limpio no vacío: {len(df) - df['texto_vacio'].sum()}")
print(f"Columnas disponibles: {list(df.columns)}")

=== RESUMEN PREPROCESAMIENTO ===
Total registros: 20001
Idiomas: español=20001, portugués=0, desconocido=0
Mensajes con texto limpio no vacío: 20001
Columnas disponibles: ['session_id', 'turn_number', 'flow_name', 'usuario', 'fecha', 'intencion', 'nivel_frustracion', 'texto_espanol', 'texto_portugues', 'es_churn_risk', 'resolved', 'idioma', 'texto_original', 'texto_clean', 'texto_vacio']
